# Cleaning decisions and implementation

Every issue found in `01_explore.ipynb` gets a documented decision here,
followed by the code that applies it.

Input:  `data/raw/afad_2011_2026.csv`  (never modified)
Output: `data/processed/afad_clean.csv`

### Decision — Issue 1: `Date` stored as text

**Problem**
`Date` is read as a string. The format is `DD/MM/YYYY HH:MM:SS` — day
first, confirmed against the raw file.

**Decision**
Parse to a datetime type, explicitly declaring day-first. Count and report
any rows that fail to parse.

**Reasoning**
Without an explicit day-first declaration, a parser may read `09/03/2024`
as 9 March instead of 3 September. For days 1–12 this produces a wrong
date with no error raised — a silent failure, which is the kind that
survives into the published result.

**Verification**
The number of failed parses is printed. Expected: 0. If it is not 0, the
failing rows are inspected before going further.

### Decision — Issue 2: magnitude scales

**Problem**
The catalog mixes magnitude scales in two ways. Within 2011–2013 both `Md`
and `Ml` appear in the same year, so those years are not internally
homogeneous. Across years the mix shifts from 33% Md (2011) to ~0% (2014+),
which is what breaks any comparison over time: a 2.5 in 2012 and a 2.5 in
2024 are not the same measurement.

**Options considered**
- **A** — drop 2011–2013 entirely. Clean, but discards three years when a
  consistent ML series already exists in all of them.
- **B** — convert Md to ML with a published relation. Rejected: conversion
  relations are region-specific and I cannot demonstrate their accuracy to
  the reader with my own data. An assumption I cannot verify is worse than
  no assumption.
- **C** — keep `Type` as a column; merge only case variants. *(chosen)*
- **D** — drop Md records and present the remainder as the yearly count.
  Rejected: this produces the same rows as C but labels them as total
  seismicity. 2011 would read 8,289 instead of 12,405 — an undercount
  presented as a real count.

**Decision — C**
Keep `Type` in the dataset. Merge `Ml`→`ML` and `MW`→`Mw`, since those are
spelling variants of the same scale, not different measurements. Never merge
`Md` into `ML`. Every analysis states which scale it covers.

**Known limitation**
The ML series exists in all 16 years, but in 2011 it represents only 67% of
that year's records and in 2012 about 90% — the rest were Md. Plotting the
ML series alone would therefore show an artificially low 2011–2012 and a
false upward trend.

Those two years are excluded from the trend comparison, and the audit
notebook reports their Md share explicitly so the exclusion is visible
rather than silent.

### Decision — Issue 9: records outside Turkey

**Problem**
26,672 records (5.5%) fall outside roughly 36–42°N / 26–45°E. Not a data
error — AFAD monitors the wider region, so events in Greece, Georgia,
Iran and Iraq are recorded deliberately.

**Options considered**
- **Filter them out.** A "Turkey" claim becomes clean, but the cost is
  losing external validation: my 2020 count (33,827) matches AFAD's
  official figure (33,824) precisely *because* both include foreign
  events. Filtering drops it to ~32,000 and it can no longer be checked
  against any published number.
- **Keep and flag.** *(chosen)*
- **Keep and ignore.** Least work, but then "earthquakes in Turkey" would
  be a misleading label.

**Decision**
Keep every record. Add a boolean column `in_turkey_box`, true when
latitude is within 36–42 and longitude within 26–45.

Named `in_turkey_box`, not `in_turkey`: Turkey is not a rectangle, and
points in the Aegean can sit inside the box while lying in Greek waters.
The column claims only what it measures.

**Reasoning**
Reversible. External validation is preserved, and any analysis that needs
a narrower scope can filter on the flag. Filtering at cleaning time would
have thrown that away permanently.

**Obligation this creates**
Since foreign records are retained, no chart or sentence may say
"earthquakes in Turkey". The correct phrasing is "records in the AFAD
catalog".

### Decision — Issue 5: default depth value (7.0 km)

**Decision**
Add a boolean column `depth_is_default` (true where `Depth == 7.0`).
Leave the values untouched. `Depth` is not used in any analysis.

**Reasoning**
Two independent reasons:
1. 37.5% of the values are a placeholder, not a measurement, and they do
   not look missing — any mean, histogram or correlation over `Depth`
   would silently include 180,700 fabricated values.
2. The research question is whether the *number* of recorded earthquakes
   is rising. Depth does not enter that question at all.

A column being present is not a reason to use it.

**Still reported**
The 37.5% figure remains a headline finding of the audit. Not using a
column in the analysis is different from not reporting what is wrong
with it.

### Decision — Issue 6: negative depth values

**Decision:** No action beyond flagging. Covered by Decision 5.
**Reasoning:** 4 records with physically impossible depth (min −0.034 km),
numerical artefacts of the location solution. Since `Depth` is unused,
they affect nothing. Recorded here so the reader knows they were found.

### Decision — Issue 7: magnitude exactly 0

**Decision:** No action. Records retained.

**Reasoning**
Magnitude scales are logarithmic and unbounded below — negative magnitudes
are real and routinely recorded in dense local networks, so M0 is a
physically meaningful very small event, not an impossible value.

It is unusual for this catalog (25th percentile is 1.3, so AFAD's detection
floor sits around M1–1.5), but unusual is not invalid.

More importantly, the completeness threshold applied in the analysis sits
well above 0, so these 12 records are excluded by a filter that already
exists. Adding a deletion rule would change nothing and add a rule.

**Flagged, not deleted:** the 12 records are reported in the audit.

### Decision — Issue 11: incomplete first and last years

**Problem**
The catalog starts 17 Sep 2011 and ends 19 Sep 2026. 2011 covers ~3.5
months, 2026 covers ~8.7 months. Neither is comparable to a full year.

**Options considered**
- **A** — exclude partial years from the trend comparison. *(chosen)*
- **B** — convert counts to a daily rate and annualise. Rejected: this
  assumes the remaining 3.5 months of 2026 resemble the first 8.7. For
  earthquake data that assumption is especially weak — a single mainshock
  can double a year's count, as 2023 and 2020 both show. Trading a data
  point for an unverifiable assumption is the wrong direction.

**Decision**
The trend comparison covers **2013–2025** — thirteen complete years.
2011 and 2012 are already excluded under Decision 2 (magnitude scale),
2026 is excluded here.

Nothing is deleted. 2026 remains in the dataset and its partial count is
reported as a footnote: 27,454 records as of 19 Sep 2026.

**Obligation this creates**
Every chart and sentence about the trend states its window explicitly —
"2013–2025, full years only" — never "up to 2026", which would leave the
reader guessing whether 2026 is in or out.

### Decision — Issue 12: incomplete months

**Problem**
Three months in the catalog are incomplete, for two different reasons:
- `2011-09` and `2026-09` — the download window starts on 17 Sep 2011 and
  ends on 19 Sep 2026, so both months are cut short by my own query.
- `2013-12` — the download is complete, but AFAD's catalogue itself holds
  only 324 records for the month (the rest of 2013 averages 2,117) and none
  at all on 31 December. Details and verification in
  `docs/data_quality_notes.md`.

**Decision**
Keep every record. Add three columns:
- `ym` — year-month (`YYYY-MM`), used for all monthly analyses.
- `month_incomplete` — true for records in the three months above.
- `incomplete_reason` — `download_window`, `catalogue_gap` or `complete`.

**Reasoning**
Deleting the months would lose real records; leaving them unmarked means a
later reader takes them at face value and never learns what is missing.

The flag is set per month, not per year. A year's completeness can be
derived from its months (twelve present, none flagged), but a month's
cannot be derived from a year flag — a year-level flag would force dropping
the complete months Oct–Dec 2011 from any monthly chart.

The reason column exists because the two kinds of gap are treated
differently: `download_window` months say something about my query, not the
data, so they are dropped from charts; the `catalogue_gap` month is a
finding, so it stays visible.

`complete` is used instead of an empty string because an empty string
written to CSV is read back as `NaN`, which silently breaks comparisons
such as `== ""`.

**Relation to Decisions 2 and 11**
Yearly comparisons still use full years only (Decision 11). Monthly
analyses drop only the flagged months. Any comparison that depends on
magnitude values starts in 2013, since 2011–2012 mix Md and ML (Decision 2).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("../data/raw/afad_2011_2026.csv")
print(df.head())
df.info()

                  Date  Longitude  Latitude  Depth   Rms Type  Magnitude  \
0  19/09/2026 20:24:25   39.21700  38.45767   7.12  0.30   ML        2.6   
1  19/09/2026 20:12:32   26.94750  38.77317   6.61  0.35   ML        2.1   
2  19/09/2026 20:00:57   26.95883  38.75217   6.80  0.40   ML        2.3   
3  19/09/2026 19:55:18   36.36167  37.97867   7.05  0.28   ML        2.1   
4  19/09/2026 19:50:34   28.13317  39.12100   6.99  0.40   ML        1.3   

                 Location  EventID  
0        Sivrice (Elazığ)   729077  
1          Aliağa (İzmir)   729075  
2          Aliağa (İzmir)   729074  
3  Göksun (Kahramanmaraş)   729072  
4    Sındırgı (Balıkesir)   729076  
<class 'pandas.DataFrame'>
RangeIndex: 481591 entries, 0 to 481590
Data columns (total 9 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   Date       481591 non-null  str    
 1   Longitude  481591 non-null  float64
 2   Latitude   481591 non-null  float64
 3   Depth      4

In [2]:
def parse_dates(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    failed = df["Date"].isna().sum()
    print("Unparsed dates:", failed)
    return df

In [3]:
df = parse_dates(df)

Unparsed dates: 0


In [4]:
def normalise_scale_labels(df):
    df = df.copy()
    changed = df["Type"].isin(["Ml", "MW"]).sum()
    df["Type"] = df["Type"].replace({"Ml": "ML", "MW": "Mw"})
    print("Labels normalised:", changed)
    return df

In [5]:
df = normalise_scale_labels(df)
print(df["Type"].value_counts())

Labels normalised: 57858
Type
ML        465991
Mw          8016
Md          7576
Mwp            6
Ms(BB)         2
Name: count, dtype: int64


In [6]:
def flag_default_depth(df):
    df = df.copy()
    df["depth_is_default"] = df["Depth"] == 7.0
    print("Default depth records:", df["depth_is_default"].sum())
    return df

In [7]:
df = flag_default_depth(df)

Default depth records: 180700


In [8]:
def flag_in_turkey_box(df):
    df = df.copy()
    df["in_turkey_box"] = ((df["Latitude"].between(36, 42) & (df["Longitude"].between(26, 45))))
    print("Records outside box:", (~df["in_turkey_box"]).sum())
    return df

In [9]:
df = flag_in_turkey_box(df)

Records outside box: 26672


In [10]:
def add_year(df):
    df = df.copy()
    df["year"] = df["Date"].dt.year
    print(df.shape)
    return df

In [11]:
df = add_year(df)

(481591, 12)


In [12]:
def flag_incomplete_periods(df):
    df = df.copy()
    reasons = {
        "2011-09": "download_window",
        "2013-12": "catalogue_gap",
        "2026-09": "download_window",
    }
    df["ym"] = df["Date"].dt.strftime("%Y-%m")
    df["month_incomplete"] = df["ym"].isin(reasons)
    df["incomplete_reason"] = df["ym"].map(reasons).fillna("complete")
    print("Flagged records:", df["month_incomplete"].sum())
    return df

In [13]:
df = flag_incomplete_periods(df)

Flagged records: 2754


In [14]:
df.to_csv("../data/processed/afad_clean.csv", index=False)

## Validation

Read the saved file back and check it, so the checks run against what a reader of this project will actually open, not against the variable in memory.

In [15]:
check = pd.read_csv("../data/processed/afad_clean.csv")
print(check.shape)
print(check.dtypes)

(481591, 15)
Date                     str
Longitude            float64
Latitude             float64
Depth                float64
Rms                  float64
Type                     str
Magnitude            float64
Location                 str
EventID                int64
depth_is_default        bool
in_turkey_box           bool
year                   int64
ym                       str
month_incomplete        bool
incomplete_reason        str
dtype: object


In [16]:
def validate(df):
    assert df.shape[0] == 481591, "satır sayısı değişmiş"
    assert df["Magnitude"].between(-5,10).all(), "büyüklük aralık dışı"
    assert df["Latitude"].between(-90,90).all(), "enlem aralık dışı"
    assert df["Longitude"].between(-180,180).all(), "boylam aralık dışı"
    assert df["depth_is_default"].sum() == 180700, "varsayılan derinlik sayısı değişmiş"
    assert (~df["in_turkey_box"]).sum() == 26672, "kutu dışı sayısı değişmiş"
    print("Tüm kontroller geçti")

validate(check)

Tüm kontroller geçti


### Known-event checks

The range checks above only prove the file is internally consistent. These compare it with events known from outside the data.

In [17]:
big = check[check["Magnitude"] >= 7.0]
print(big[["Date", "Magnitude", "Location", "Type"]])

                       Date  Magnitude                           Location  \
184606  2023-02-06 13:24:47        7.6           Elbistan (Kahramanmaraş)   
184852  2023-02-06 04:17:32        7.7           Pazarcık (Kahramanmaraş)   
317392  2017-11-12 21:18:14        7.2  Sarpol-e-Zahab, Kermanshah (İran)   

          Type  
184606      Mw  
184852      Mw  
317392  Ms(BB)  


In [18]:
print(check.loc[317392, "in_turkey_box"])

False


In [19]:
print(check["Date"].min(), check["Date"].max())

2011-09-17 21:51:54 2026-09-19 20:24:25


In [20]:
van = check[check["Date"].str[:10] == "2011-10-23"]
print(van.nlargest(5, "Magnitude")[["Date", "Magnitude", "Location", "Type"]])

                       Date  Magnitude                           Location Type
479613  2011-10-23 13:41:20        6.7                        Tuşba (Van)   ML
479431  2011-10-23 23:45:34        5.8  Van Gölü - [06.41 km] Tuşba (Van)   ML
479609  2011-10-23 13:56:48        5.8                        Tuşba (Van)   ML
479600  2011-10-23 14:32:40        5.5                        Tuşba (Van)   ML
479608  2011-10-23 14:00:29        5.3  Van Gölü - [07.20 km] Tuşba (Van)   ML
